In [1]:
import sys, os
import traceback
from datetime import datetime
sys.path.append(r'C:/data/EnergyTrading/Python/')
# local imports
from Loaders.fund_fetch import fetch_fund_ftp
from Database.DB_reader import Database


In [2]:
from Utilities.email_sending import send_plain_email, send_html_email
EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD') # the password needs to be set as EMAIL_PASSWORD in system variables of the computer where the process is running, it is located in S:/Algo/email_password.txt
if EMAIL_PASSWORD is None:
    raise ValueError("EMAIL_PASSWORD environment variable not set")

RECIPIENT = ["scasny_martin@energytrading.sk", "krajcovic_matej@energytrading.sk"] # can also be a list of recipients

ValueError: EMAIL_PASSWORD environment variable not set

In [ ]:
if __name__ == "__main__":
    db = Database()
    # from_ = datetime(2024,10,18).date()
    from_ = datetime.now().date()
    try:
        result = {
            **fetch_fund_ftp(from_= from_, fund_list=['AvailCap','InstCap', 'LtInstCap'],
                      grid_list=['DEU', 'FRA'],
                      hour='12'),
            **fetch_fund_ftp(from_= from_, fund_list=['ResidualDemand'],
                                grid_list=['DEU', 'FRA', 'NLD', 'BEL', 'AUT', 'CZE'],
                                hour='00'),
            **fetch_fund_ftp(from_= from_, fund_list=['CON_mnd', 'Wind_mnd', 'Solar_mnd'],
                                grid_list=['DEU', 'FRA', 'NLD', 'BEL', 'AUT', 'ROU'],
                                hour='00'),
            **fetch_fund_ftp(from_= from_, fund_list=['INF'],
                          grid_list=['AUT', 'FRA'],
                          hour='00'),
            **fetch_fund_ftp(from_ = from_, fund_list=['Wind', 'Solar'],
                             grid_list= ['ROU'],
                            hour='00'),
            **fetch_fund_ftp(from_= from_, fund_list=['Temp', 'Temp_mnd'],
                  grid_list=['HUN', 'DEU', 'FRA'],
                  hour='00')
        }

        # Capture the traceback
        error_traceback = traceback.format_exc()
        # Format dictionary items for HTML content
        items_html = '\n'.join([f'<p>{key}: {value}</p>' for key, value in result.items()])
        html_content = f"""\
        <html>
            <body>
                <h1>Data updated</h1>
                <p>{items_html}</p>
            </body>
        </html>
        """

        send_html_email(
            RECIPIENT, 
            "AUTOMATIC JOBS - REPORT - fund_daily_update", 
            "", 
            html_content, 
            "",
            email_password=EMAIL_PASSWORD
        )
    except Exception as e:
        error_message = str(e)

        # Capture the traceback
        error_traceback = traceback.format_exc()
        html_content = f"""\
        <html>
            <body>
                <h1>Failed job fund_daily_update</h1>
                <p>{error_message}</p>
                <pre>{error_traceback}</pre>
            </body>
        </html>
        """

        send_html_email(
            RECIPIENT, 
            "AUTOMATIC JOBS - FAILED - fund_daily_update", 
            "", 
            html_content, 
            "",
            email_password=EMAIL_PASSWORD
        )
    
